In [2]:
!pip install earthengine-api geemap geopandas folium -q

In [3]:
import ee
import geemap
import geopandas as gpd
import folium
print("All libraries loaded successfully!")

All libraries loaded successfully!


In [11]:
import ee
import geemap

In [13]:
ee.Authenticate()
ee.Initialize(project='tamil-nadu-flood-risk')

In [14]:
print(ee.String('Tamil Nadu flood mapping — ready!').getInfo())

Tamil Nadu flood mapping — ready!


Install all libraries we need

In [15]:
!pip install geemap folium geopandas rasterstats contextily -q

In [16]:
# Cell 4 — Sentinel-1 SAR flood map over Tamil Nadu
import ee
import geemap

ee.Initialize(project='tamil-nadu-flood-risk')

# Tamil Nadu bounding box
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# Load Sentinel-1 SAR collection
# October-December = Tamil Nadu's northeast monsoon flood season
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(tamil_nadu) \
    .filterDate('2023-10-01', '2023-12-31') \
    .select('VV')

# How many images found?
print('Sentinel-1 images found:', s1.size().getInfo())

# Median composite across the season
s1_median = s1.median().clip(tamil_nadu)

# Classify flood pixels
# VV backscatter below -15 dB = likely water/flood
flood_mask = s1_median.lt(-15)

# Display map
Map = geemap.Map()
Map.centerObject(tamil_nadu, zoom=7)

# Add SAR layer
Map.addLayer(
    s1_median,
    {'min': -25, 'max': 0, 'palette': ['#1a6faf', 'white', '#2d8a4e']},
    'Sentinel-1 VV backscatter'
)

# Add flood mask — red = flooded pixels
Map.addLayer(
    flood_mask.updateMask(flood_mask),
    {'palette': ['#ff0000']},
    'Flood pixels (VV < -15dB)'
)

Map

Sentinel-1 images found: 96


Map(center=[10.749705958455543, 78.24999999999999], controls=(WidgetControl(options=['position', 'transparent_…

In [17]:
# Cell 5 — Add Indian district boundaries + flood statistics
import ee
import geemap

ee.Initialize(project='tamil-nadu-flood-risk')

# Tamil Nadu boundary
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

# Load Sentinel-1 — same as before
s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(tamil_nadu) \
    .filterDate('2023-10-01', '2023-12-31') \
    .select('VV')

s1_median = s1.median().clip(tamil_nadu)

# Flood mask
flood_mask = s1_median.lt(-15)

# Load India district boundaries from GEE dataset
districts = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Tamil Nadu'))

print('Number of Tamil Nadu districts:', districts.size().getInfo())

# Calculate flood pixel count per district
flood_stats = flood_mask.reduceRegions(
    collection=districts,
    reducer=ee.Reducer.sum(),
    scale=100  # 100 metre resolution
)

# Display map
Map = geemap.Map()
Map.centerObject(tamil_nadu, zoom=7)

# SAR background
Map.addLayer(
    s1_median,
    {'min': -25, 'max': 0, 'palette': ['#1a6faf', 'white', '#2d8a4e']},
    'Sentinel-1 VV'
)

# Flood pixels
Map.addLayer(
    flood_mask.updateMask(flood_mask),
    {'palette': ['#ff0000']},
    'Flood pixels'
)

# District boundaries
Map.addLayer(
    districts,
    {'color': 'yellow', 'fillColor': '00000000'},
    'Tamil Nadu Districts'
)

Map

Number of Tamil Nadu districts: 29


Map(center=[10.749705958455543, 78.24999999999999], controls=(WidgetControl(options=['position', 'transparent_…

In [19]:
# Cell 6 — Fixed version
import ee
import geemap
import pandas as pd

ee.Initialize(project='tamil-nadu-flood-risk')

# ── 1. School data ──
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}

schools_df = pd.DataFrame(data)
print(f'Schools loaded: {len(schools_df)}')

# ── 2. Convert to GEE FeatureCollection ──
features = []
for _, row in schools_df.iterrows():
    feature = ee.Feature(
        ee.Geometry.Point([row['longitude'], row['latitude']]),
        {
            'school_name': row['school_name'],
            'connectivity': row['connectivity']
        }
    )
    features.append(feature)

schools_ee = ee.FeatureCollection(features)

# ── 3. Tamil Nadu + Sentinel-1 ──
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(tamil_nadu) \
    .filterDate('2023-10-01', '2023-12-31') \
    .select('VV')

s1_median = s1.median().clip(tamil_nadu)
flood_mask = s1_median.lt(-15)

districts = ee.FeatureCollection("FAO/GAUL/2015/level2") \
    .filter(ee.Filter.eq('ADM1_NAME', 'Tamil Nadu'))

# ── 4. Map ──
Map = geemap.Map()
Map.centerObject(tamil_nadu, zoom=7)

Map.addLayer(
    s1_median,
    {'min': -25, 'max': 0, 'palette': ['#1a6faf', 'white', '#2d8a4e']},
    'Sentinel-1 VV'
)

Map.addLayer(
    flood_mask.updateMask(flood_mask),
    {'palette': ['#ff0000']},
    'Flood pixels'
)

Map.addLayer(
    districts,
    {'color': 'yellow', 'fillColor': '00000000'},
    'Districts'
)

# Connected schools — green
connected = schools_ee.filter(ee.Filter.eq('connectivity', 'connected'))
Map.addLayer(connected, {'color': '00ff00'}, 'Schools connected')

# No connectivity schools — orange
no_conn = schools_ee.filter(ee.Filter.eq('connectivity', 'none'))
Map.addLayer(no_conn, {'color': 'ff6600'}, 'Schools no connectivity')

Map

Schools loaded: 15


Map(center=[10.749705958455543, 78.24999999999999], controls=(WidgetControl(options=['position', 'transparent_…

In [25]:
# Cell 7 — Extract flood exposure for each school
import ee
import geemap
import pandas as pd

ee.Initialize(project='tamil-nadu-flood-risk')

# ── 1. Rebuild schools and flood mask ──
data = {
    'school_name': [
        'Govt High School Chennai', 'Govt School Cuddalore',
        'Panchayat School Nagapattinam', 'Govt School Thanjavur',
        'High School Coimbatore', 'Govt School Madurai',
        'Panchayat School Tirunelveli', 'Govt School Salem',
        'High School Vellore', 'Govt School Tiruchirappalli',
        'School Ramanathapuram', 'School Puducherry Border',
        'School Kanchipuram', 'School Villupuram',
        'School Tuticorin'
    ],
    'latitude': [
        13.08, 11.75, 10.76, 10.78, 11.01,
        9.93, 8.71, 11.65, 12.92, 10.79,
        9.37, 11.93, 12.83, 11.93, 8.80
    ],
    'longitude': [
        80.27, 79.75, 79.84, 79.13, 76.96,
        78.12, 77.69, 78.16, 79.13, 78.68,
        78.83, 79.83, 79.70, 79.49, 78.15
    ],
    'connectivity': [
        'connected', 'none', 'none',
        'connected', 'connected', 'connected',
        'none', 'connected', 'connected',
        'none', 'none', 'connected',
        'none', 'none', 'connected'
    ]
}

schools_df = pd.DataFrame(data)

# ── 2. Build GEE FeatureCollection with 1km buffer ──
features = []
for _, row in schools_df.iterrows():
    point = ee.Geometry.Point([row['longitude'], row['latitude']])
    buffer = point.buffer(1000)  # 1km buffer around each school
    feature = ee.Feature(buffer, {
        'school_name': row['school_name'],
        'connectivity': row['connectivity'],
        'latitude': row['latitude'],
        'longitude': row['longitude']
    })
    features.append(feature)

schools_buffered = ee.FeatureCollection(features)

# ── 3. Rebuild flood mask ──
tamil_nadu = ee.Geometry.Rectangle([76.0, 8.0, 80.5, 13.5])

s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filter(ee.Filter.eq('instrumentMode', 'IW')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.eq('orbitProperties_pass', 'DESCENDING')) \
    .filterBounds(tamil_nadu) \
    .filterDate('2023-10-01', '2023-12-31') \
    .select('VV')

s1_median = s1.median().clip(tamil_nadu)

# VV backscatter value at each school (lower = more water)
# Also compute flood pixel percentage within 1km buffer
flood_mask = s1_median.lt(-15).rename('flood_pixel')

# ── 4. Extract mean VV backscatter per school buffer ──
vv_stats = s1_median.rename('vv_mean').reduceRegions(
    collection=schools_buffered,
    reducer=ee.Reducer.mean(),
    scale=100
)

# Extract % of flood pixels within each school buffer
flood_stats = flood_mask.reduceRegions(
    collection=schools_buffered,
    reducer=ee.Reducer.mean(),
    scale=100
)

# ── 5. Export results ──
# Convert to pandas
vv_list = vv_stats.select(['school_name', 'connectivity', 'mean']).getInfo()
flood_list = flood_stats.select(['school_name', 'mean']).getInfo()

# Parse results
vv_data = [f['properties'] for f in vv_list['features']]
flood_data = [f['properties'] for f in flood_list['features']]

vv_df = pd.DataFrame(vv_data)
vv_df = vv_df.rename(columns={'mean': 'vv_mean'}) # Rename 'mean' from reducer output to 'vv_mean'
flood_df = pd.DataFrame(flood_data)

# Merge
results = vv_df.merge(flood_df, on='school_name')
results['flood_exposure_pct'] = (results['mean'] * 100).round(2)
results['vv_mean'] = results['vv_mean'].round(2)

# Risk tier
def risk_tier(pct):
    if pct >= 30:
        return 'HIGH'
    elif pct >= 10:
        return 'MEDIUM'
    else:
        return 'LOW'

results['risk_tier'] = results['flood_exposure_pct'].apply(risk_tier)

# Sort by flood exposure
results = results.sort_values('flood_exposure_pct', ascending=False)
# Fixed last line — Cell 7
print('\n=== TAMIL NADU SCHOOL FLOOD RISK RESULTS ===\n')
cols = ['school_name', 'connectivity', 'flood_exposure_pct', 'vv_mean', 'risk_tier']
print(results[cols].to_string(index=False))


=== TAMIL NADU SCHOOL FLOOD RISK RESULTS ===

                  school_name connectivity  flood_exposure_pct  vv_mean risk_tier
     School Puducherry Border    connected               10.87    -2.42    MEDIUM
        School Ramanathapuram         none                1.59    -4.27       LOW
        Govt School Cuddalore         none                1.56    -5.91       LOW
Panchayat School Nagapattinam         none                1.07    -6.40       LOW
          Govt School Madurai    connected                0.95    -5.05       LOW
          High School Vellore    connected                0.63    -6.20       LOW
            School Villupuram         none                0.63    -5.54       LOW
             School Tuticorin    connected                0.34    -4.99       LOW
  Govt School Tiruchirappalli         none                0.34    -7.59       LOW
        Govt School Thanjavur    connected                0.00    -7.02       LOW
     Govt High School Chennai    connected         

In [26]:
# Cell 8 — Risk map with colour coded schools
import folium
import pandas as pd

# Rebuild results dataframe with risk data we just got
results_data = {
    'school_name': [
        'School Puducherry Border', 'School Ramanathapuram',
        'Govt School Cuddalore', 'Panchayat School Nagapattinam',
        'Govt School Madurai', 'High School Vellore',
        'School Villupuram', 'School Tuticorin',
        'Govt School Tiruchirappalli', 'Govt School Thanjavur',
        'Govt High School Chennai', 'Panchayat School Tirunelveli',
        'High School Coimbatore', 'Govt School Salem',
        'School Kanchipuram'
    ],
    'latitude': [
        11.93, 9.37, 11.75, 10.76, 9.93,
        12.92, 11.93, 8.80, 10.79, 10.78,
        13.08, 8.71, 11.01, 11.65, 12.83
    ],
    'longitude': [
        79.83, 78.83, 79.75, 79.84, 78.12,
        79.13, 79.49, 78.15, 78.68, 79.13,
        80.27, 77.69, 76.96, 78.16, 79.70
    ],
    'connectivity': [
        'connected', 'none', 'none', 'none',
        'connected', 'connected', 'none', 'connected',
        'none', 'connected', 'connected', 'none',
        'connected', 'connected', 'none'
    ],
    'flood_exposure_pct': [
        10.87, 1.59, 1.56, 1.07, 0.95,
        0.63, 0.63, 0.34, 0.34, 0.00,
        0.00, 0.00, 0.00, 0.00, 0.00
    ],
    'risk_tier': [
        'MEDIUM', 'LOW', 'LOW', 'LOW', 'LOW',
        'LOW', 'LOW', 'LOW', 'LOW', 'LOW',
        'LOW', 'LOW', 'LOW', 'LOW', 'LOW'
    ]
}

df = pd.DataFrame(results_data)

# ── Build Folium map ──
m = folium.Map(
    location=[10.5, 78.5],
    zoom_start=7,
    tiles='CartoDB positron'
)

# Colour logic
def get_color(row):
    if row['risk_tier'] == 'HIGH':
        return 'red'
    elif row['risk_tier'] == 'MEDIUM':
        return 'orange'
    else:
        # Low risk — green if connected, gray if no connectivity
        return 'green' if row['connectivity'] == 'connected' else 'gray'

# Add each school as a circle marker
for _, row in df.iterrows():
    color = get_color(row)

    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=10,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.8,
        popup=folium.Popup(
            f"""
            <b>{row['school_name']}</b><br>
            Connectivity: {row['connectivity']}<br>
            Flood exposure: {row['flood_exposure_pct']}%<br>
            Risk tier: <b>{row['risk_tier']}</b>
            """,
            max_width=250
        ),
        tooltip=row['school_name']
    ).add_to(m)

# Add legend
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px;
     background: white; padding: 12px; border-radius: 8px;
     border: 1px solid grey; font-size: 13px; z-index: 1000;">
     <b>School Risk Legend</b><br>
     <span style="color:red">●</span> High flood risk<br>
     <span style="color:orange">●</span> Medium flood risk<br>
     <span style="color:green">●</span> Low risk — connected<br>
     <span style="color:gray">●</span> Low risk — no connectivity<br>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Save
m.save('tamil_nadu_school_risk.html')
print('Map saved! Open tamil_nadu_school_risk.html to view')

# Display in Colab
m

Map saved! Open tamil_nadu_school_risk.html to view
